### Feature engineering

This notebook turns the raw columns into model-ready features. Every value needed during prediction must be created from information available at the time the load is posted.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data = pd.read_csv(ROOT / "data" / "train-test.csv", parse_dates=["date"])

cutoff = pd.Timestamp("2025-09-01")
development = data.loc[data.date < cutoff].copy()
holdout = data.loc[data.date >= cutoff].copy()

### Split

January through August is used to fit the preprocessing rules. September and October are transformed with those saved rules but are not used to calculate medians or city coordinate replacements. This reproduces what happens when the trained pipeline receives future loads.

In [2]:
pd.DataFrame({
    "rows": [len(development), len(holdout)],
    "date_min": [development.date.min(), holdout.date.min()],
    "date_max": [development.date.max(), holdout.date.max()],
}, index=["development", "holdout"])

,rows,date_min,date_max
development,38477,2025-01-01,2025-08-31
holdout,9523,2025-09-01,2025-10-31


### Imputation

The EDA showed that negative weights have the same range as normal weights after removing the sign, so they are treated as entry errors and corrected with `abs()`. Missing numeric values are filled with medians from the earlier training period only. Flags are kept for missing or corrected values so CatBoost can learn whether those rows behave differently.

In [3]:
numeric_columns = [
    "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon",
    "distance", "weight", "market_index", "quote_signal",
]
# Fit cleaning values on development data only.
development_numeric = development[numeric_columns].copy()
development_numeric["weight"] = development_numeric.weight.abs()
numeric_medians = development_numeric.median()
numeric_medians.to_frame("development_median")

,development_median
pickup_lat,35.29479
pickup_lon,-88.08915
delivery_lat,35.29479
delivery_lon,-87.52871
distance,954.80000
weight,31472.50000
market_index,1.12000
quote_signal,2.05361


### Transformations

The original business columns are retained and several useful views are added. Calendar features represent the date, lane features represent the pickup-delivery pair, and geographic features describe route direction and straight-line distance. Log versions of distance and weight make large and small values easier for the trees to separate.

In [4]:
EARTH_RADIUS_MILES = 3_958.8
categorical_columns = ["pickup", "delivery", "equipment", "lane"]

# Straight-line distance between two coordinate pairs, in miles.
def haversine(frame):
    lat1 = np.radians(frame.pickup_lat.to_numpy())
    lon1 = np.radians(frame.pickup_lon.to_numpy())
    lat2 = np.radians(frame.delivery_lat.to_numpy())
    lon2 = np.radians(frame.delivery_lon.to_numpy())
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    value = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
    )
    return 2 * EARTH_RADIUS_MILES * np.arcsin(np.sqrt(np.clip(value, 0, 1)))

def build_features(frame, medians):
    numeric = frame[numeric_columns].copy()
    # Keep the original data-quality state after imputation.
    flags = pd.DataFrame(index=frame.index)
    flags["weight_was_missing"] = numeric.weight.isna().astype("int8")
    flags["weight_sign_corrected"] = numeric.weight.lt(0).astype("int8")
    flags["market_index_was_missing"] = numeric.market_index.isna().astype("int8")
    numeric["weight"] = numeric.weight.abs()
    features = pd.concat([numeric.fillna(medians), flags], axis=1)
    date = frame.date

    features["pickup"] = frame.pickup.astype(str)
    features["delivery"] = frame.delivery.astype(str)
    features["equipment"] = frame.equipment.astype(str)
    features["lane"] = features.pickup + " -> " + features.delivery

    features["month"] = date.dt.month
    features["day_of_week"] = date.dt.dayofweek
    features["day_of_year"] = date.dt.dayofyear
    features["week_of_year"] = date.dt.isocalendar().week.astype(int)
    features["days_since_2025"] = (date - pd.Timestamp("2025-01-01")).dt.days
    # Cyclical encoding keeps adjacent periods close, including year boundaries.
    features["month_sin"] = np.sin(2 * np.pi * features.month / 12)
    features["month_cos"] = np.cos(2 * np.pi * features.month / 12)
    features["dow_sin"] = np.sin(2 * np.pi * features.day_of_week / 7)
    features["dow_cos"] = np.cos(2 * np.pi * features.day_of_week / 7)
    features["doy_sin"] = np.sin(2 * np.pi * features.day_of_year / 365.25)
    features["doy_cos"] = np.cos(2 * np.pi * features.day_of_year / 365.25)

    features["haversine_distance"] = haversine(features)
    features["distance_to_haversine_ratio"] = features.distance / np.maximum(features.haversine_distance, 1)
    features["lat_delta"] = features.delivery_lat - features.pickup_lat
    features["lon_delta"] = features.delivery_lon - features.pickup_lon
    features["midpoint_lat"] = (features.pickup_lat + features.delivery_lat) / 2
    features["midpoint_lon"] = (features.pickup_lon + features.delivery_lon) / 2
    features["log_distance"] = np.log1p(features.distance.clip(lower=0))
    features["log_weight"] = np.log1p(features.weight.clip(lower=0))
    return features

x_development = build_features(development, numeric_medians)
x_holdout = build_features(holdout, numeric_medians)
y_development = development.posted_rate
y_holdout = holdout.posted_rate

### Validation

The earlier and later datasets must produce exactly the same feature columns. This check also confirms that no missing values remain and that `posted_rate` and `load_id` are not model inputs. Including either would leak the answer or let the model memorize rows.

In [5]:
pd.Series({
    "development_rows": len(x_development),
    "holdout_rows": len(x_holdout),
    "feature_count": x_development.shape[1],
    "development_missing": int(x_development.isna().sum().sum()),
    "holdout_missing": int(x_holdout.isna().sum().sum()),
    "same_schema": x_development.columns.equals(x_holdout.columns),
    "target_excluded": "posted_rate" not in x_development,
    "identifier_excluded": "load_id" not in x_development,
}, name="value").to_frame()

,value
development_rows,38477
holdout_rows,9523
feature_count,34
development_missing,0
holdout_missing,0
same_schema,True
target_excluded,True
identifier_excluded,True


In [6]:
pd.DataFrame({
    "development": x_development[["weight_was_missing", "weight_sign_corrected", "market_index_was_missing"]].sum(),
    "holdout": x_holdout[["weight_was_missing", "weight_sign_corrected", "market_index_was_missing"]].sum(),
})

,development,holdout
weight_was_missing,235,65
weight_sign_corrected,233,59
market_index_was_missing,301,73


In [7]:
pd.DataFrame({
    "feature": x_development.columns,
    "dtype": x_development.dtypes.astype(str).to_numpy(),
    "categorical": x_development.columns.isin(categorical_columns),
})

,feature,dtype,categorical
0,pickup_lat,float64,False
1,pickup_lon,float64,False
2,delivery_lat,float64,False
3,delivery_lon,float64,False
4,distance,float64,False
5,weight,float64,False
6,market_index,float64,False
7,quote_signal,float64,False
8,weight_was_missing,int8,False
9,weight_sign_corrected,int8,False


In [8]:
preview = [
    "distance", "weight", "lane", "equipment",
    "day_of_week", "doy_sin", "doy_cos",
    "haversine_distance", "distance_to_haversine_ratio",
    "log_distance", "log_weight",
]
x_development[preview].head()

,distance,weight,lane,equipment,day_of_week,doy_sin,doy_cos,haversine_distance,distance_to_haversine_ratio,log_distance,log_weight
0,274.3,30658.0,Richmond -> Baltimore,Dry Van,2,0.017202,0.999852,219.808768,1.247903,5.617861,10.330682
1,280.5,17555.0,Richmond -> Philadelphia,Reefer,2,0.017202,0.999852,220.525617,1.271961,5.640132,9.773151
2,967.8,31721.0,Philadelphia -> Green Bay,Dry Van,2,0.017202,0.999852,826.919847,1.170367,6.876058,10.364766
3,965.4,32333.0,Hartford -> Atlanta,Dry Van,2,0.017202,0.999852,840.540749,1.148546,6.873578,10.383875
4,541.9,35183.0,Dallas -> Nashville,Reefer,2,0.017202,0.999852,434.163187,1.248148,6.296925,10.468347
